In [1]:
import requests
import pandas as pd
import time
import urllib
import urllib.parse
from transformers import pipeline
import re
from dotenv import load_dotenv
import os



c:\Users\xaren\OneDrive\Documentos\Angel Sotelo\Projects\app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

load_dotenv()  # carga variables del archivo .env
LASTFM_KEY = os.getenv("LASTFM_KEY")
APP_ID = os.getenv("APP_ID")
API_KEY = os.getenv("API_KEY")
LASTFM_URL = "http://ws.audioscrobbler.com/2.0/"

headers = {
    'x-app-id': APP_ID,
    'x-api-key': API_KEY,
}

In [3]:
# Lista de artistas (puedes cambiarla)

#artists = [
    # Pop / Mainstream
#    "Taylor Swift", "Ed Sheeran", "Adele", "Olivia Rodrigo", "Billie Eilish",

    # Rock / Alternative
#    "Coldplay", "Arctic Monkeys", "Radiohead", "The Killers", "Imagine Dragons",

    # Hip-Hop / Rap
#    "Drake", "Kendrick Lamar", "Travis Scott", "J. Cole", "Post Malone",

    # Latino / Urbano
#    "Bad Bunny", "Feid", "Karol G", "Rauw Alejandro", "J Balvin",

    # Indie / Emocional
#    "Lana Del Rey", "Phoebe Bridgers", "Clairo", "Joji", "The 1975",

    # Pop / R&B Mix
#    "Harry Styles", "Dua Lipa", "Shawn Mendes", "SZA", "Frank Ocean"
#]
artists = [
    # Electrónica / Experimental
#    "Aphex Twin", "Flume", "ODESZA", "Skrillex", "Four Tet",

    # Jazz / Soul / Clásicos modernos
#    "Miles Davis", "Nina Simone", "Erykah Badu", "D'Angelo", "Norah Jones",

    # Rock clásico / legado
#    "The Beatles", "Pink Floyd", "Led Zeppelin", "Queen", "The Rolling Stones",

    # Regional / Folk / World
#    "Caetano Veloso", "Mercedes Sosa", "Buena Vista Social Club", "Ali Farka Touré", "Lila Downs",

    # K-Pop / J-Pop / Asia
    "BTS" #"BLACKPINK", "IU", "Hikaru Utada", "YOASOBI",

    # Metal / Hard
    #"Metallica", "Slipknot", "Iron Maiden", "Deftones", "Bring Me The Horizon",

    # Alternativo moderno (menos mainstream que tu lista original)
    #"Tame Impala", "Glass Animals", "alt-J", "Beach House", "Mac DeMarco",

    # Hip-hop alternativo / underground
    #"MF DOOM", "A Tribe Called Quest", "Freddie Gibbs", "Run The Jewels", "Joey Bada$$",

    # Rock en español (clásico + alternativo)
    #"Soda Stereo", "Gustavo Cerati", "Héroes del Silencio", "Caifanes", "Zoé",

    # Indie / Alternativo latino
    #"Little Jesus", "Porter", "El Mató a un Policía Motorizado", "Vetusta Morla", "Love of Lesbian",

    # Folk / Autor / Letras profundas
    #"Silvio Rodríguez", "Jorge Drexler", "Natalia Lafourcade", "Kevin Kaarl", "Ed Maverick",

    # Regional / Tradicional moderno
    #"Carla Morrison", "Mon Laferte", "Lila Downs", "Julieta Venegas", "Ximena Sariñana",

    # Rap / Hip-hop en español (menos mainstream)
    #"Canserbero", "Kase.O", "Nach", "Wos", "Nathy Peluso",

    # Pop alternativo en español
    #"Miranda!", "Dorian", "Belanova", "Reik", "Camilo Séptimo",

    # Fusión / Experimental latino
    #"Bomba Estéreo", "Calle 13", "iLe", "Nicola Cruz", "Chancha Vía Circuito"
]

In [4]:
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 33371.37it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def get_emotion_scores(text):
    result = emotion_model(
        text,
        truncation=True,     # clave
        max_length=512       # límite del modelo
    )

    return {item['label']: item['score'] for item in result[0]}

In [6]:
def search_song_soundcharts(song_name, artist_name):
    """
    Busca una canción en Soundcharts por nombre y artista.
    Retorna el UUID de la canción si se encuentra, o None si no.
    """

    # Construir término de búsqueda
    term = f"{song_name} {artist_name}"
    term_encoded = urllib.parse.quote(term)  # codifica espacios y caracteres especiales

    # URL de búsqueda con parámetros
    url = f"https://customer.api.soundcharts.com/api/v2/song/search/{term_encoded}"
    params = {
        'offset': '0',
        'limit': '5',
    }

    # Hacer la solicitud
    response = requests.get(url, headers=headers, params=params)

    
    if response.status_code != 200:
        print(f" Error en búsqueda: {response.status_code}")
        return None

    data = response.json()

    track_id = data['items'][0]['uuid']

    return track_id


In [7]:
song = "Dynamite"
artist = "BTS"

track_uuid = search_song_soundcharts(song, artist)
print("Track UUID:", track_uuid)

Track UUID: 169abfe5-0827-4592-b3b1-01a78bf18d6f


In [8]:
def get_audio_features(track_id):

    if not track_id:
        return {
            "key": None,
            "mode": None,
            "tempo": None,
            "energy": None,
            "valence": None
        }

    url = f"https://customer.api.soundcharts.com/api/v2.25/song/{track_id}"

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        data = response.json()

        audio = data.get("object", {}).get("audio", {})

        return {
            "key": audio.get("key"),
            "mode": audio.get("mode"),
            "tempo": audio.get("tempo"),
            "energy": audio.get("energy"),
            "valence": audio.get("valence")
        }

    except Exception as e:
        print(f"Error audio features: {e}")
        return {
            "key": None,
            "mode": None,
            "tempo": None,
            "energy": None,
            "valence": None
        }

In [9]:

# 🎧 Función: obtener top canciones de un artista
def get_top_tracks(artist, limit=10):
    params = {
        "method": "artist.getTopTracks",
        "artist": artist,
        "api_key": LASTFM_KEY,
        "format": "json",
        "limit": limit
    }
    
    response = requests.get(LASTFM_URL, params=params)
    data = response.json()
    
    try:
        tracks = data["toptracks"]["track"]
        return [(track["artist"]["name"], track["name"]) for track in tracks]
    except:
        return []


#  Función: obtener letra
def get_lyrics(artist, song):
    url = f"https://api.lyrics.ovh/v1/{artist}/{song}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data.get("lyrics", "")
    else:
        return ""




In [10]:
def clean_song_name(song):
    # quitar (feat. ...)
    song = re.sub(r"\(.*?\)", "", song)
    
    # quitar "feat. ..." sin paréntesis
    song = re.sub(r"feat\..*", "", song, flags=re.IGNORECASE)
    
    return song.strip()

In [ ]:
# Dataset final
dataset = []

#  Loop principal
for artist in artists:
    print(f"\nProcesando artista: {artist}")
    
    tracks = get_top_tracks(artist, limit=8)

    artist_data = [] 
    
    for artist_name, song_name in tracks:
        print(f"  → {song_name}")
        
        lyrics = get_lyrics(artist_name, song_name)
        
        # evitar canciones sin letra
        if lyrics.strip() == "":
            continue
        
        # emociones
        emotion_scores = get_emotion_scores(lyrics)

        clean_name = clean_song_name(song_name)
        # Buscar audio features
        track_id = search_song_soundcharts(clean_name, artist_name)
        print("Track ID:", track_id)
        if not track_id:
            print(" No track_id")
            continue
        audio_features = get_audio_features(track_id)
        print("Audio:", audio_features)
        if not audio_features:
            print(" No audio features")
            continue

        
        # 🔹 crear row (ANTES de usarla)

        row = {
            "artist": artist_name,
            "song": clean_name,
            "lyrics": lyrics,

            ## Emociones

            **emotion_scores,               # agrega todas las emociones

            ## De soundchart

            "key": audio_features.get("key"),
            "mode": audio_features.get("mode"),
            "tempo": audio_features.get("tempo"),
            "energy": audio_features.get("energy"),
            "valence": audio_features.get("valence"),
        }
        
        
        # append SIEMPRE fuera de condiciones
        dataset.append(row)
        
        artist_data.append(row)  
        
        time.sleep(2)  # evitar rate limit

        if artist_data:
            safe_artist = artist.replace(" ", "_").replace(".", "")
            filename = f"backup_{safe_artist}.csv"
        
            pd.DataFrame(artist_data).to_csv(filename, index=False)
        
            print(f" Backup guardado: {filename}")
        

# Crear DataFrame
df = pd.DataFrame(dataset)

# Guardar CSV
df.to_csv("songs_dataset_completo.csv", index=False)

print("\n Dataset creado: songs_dataset_completo.csv")
print(f"Total canciones: {len(df)}")


Procesando artista: BTS
  → Dynamite
Track ID: 169abfe5-0827-4592-b3b1-01a78bf18d6f
Audio: {'key': 6, 'mode': 0, 'tempo': 114.04, 'energy': 0.77, 'valence': 0.74}
 Backup guardado: backup_BTS.csv
  → Boy With Luv (feat. Halsey)
  → FAKE LOVE
Track ID: 11e86322-40a9-da6c-9dec-a0369fe50396
Audio: {'key': 2, 'mode': 0, 'tempo': 77.5, 'energy': 0.72, 'valence': 0.35}
 Backup guardado: backup_BTS.csv
  → Butter
Track ID: a72d5538-c642-11e8-b4b7-549f35161576
Audio: {'key': 8, 'mode': 1, 'tempo': 110, 'energy': 0.46, 'valence': 0.7}
 Backup guardado: backup_BTS.csv
  → DNA
Track ID: 11e826ed-e199-4834-99c4-a0369fe50396
Audio: {'key': 1, 'mode': 0, 'tempo': 129.82, 'energy': 0.78, 'valence': 0.7}
 Backup guardado: backup_BTS.csv
  → Blood Sweat & Tears
Track ID: d29af8d2-01a1-4bd6-82f8-ef78dc6dff96
Audio: {'key': 0, 'mode': 0, 'tempo': 92.9, 'energy': 0.89, 'valence': 0.61}
 Backup guardado: backup_BTS.csv
  → I NEED U
Track ID: 50362c15-6643-4be6-b2bf-134509131f1b
Audio: {'key': 5, 'mode': 0

In [12]:
#import pandas as pd
#import glob

# Buscar todos los archivos que empiezan con "backup"
#files = glob.glob("backup*")

# Leer y concatenar
#df_list = [pd.read_csv(file) for file in files]
#final_df = pd.concat(df_list, ignore_index=True)

# Guardar resultado
#final_df.to_csv("songs_dataset_completo.csv", index=False)

#print(f"Archivos concatenados: {len(files)}")